# RAG-Based Profile Matching

## Integrated Analysis Notebook

This notebook demonstrates resume processing, job matching, explainability, and performance analysis.

## Analysis Scope

This notebook covers the complete RAG workflow on 33 resumes and 5 job descriptions. It includes Part A document processing, Part B hybrid matching, explainability, and retrieval metrics.

In [ ]:
from pathlib import Path
import json
import sys
sys.path.insert(0, str(Path.cwd()))
from demo_part_a import SimplifiedRAGSystem
from job_matcher import JobDescription, JobMatcher, SearchConfig
from benchmark_metrics import run_benchmark
resume_dir = Path('sample_resumes')
jobs = json.loads(Path('job_descriptions.json').read_text(encoding='utf-8'))
print('Resumes:', len(list(resume_dir.glob('*.txt'))))
print('Jobs:', len(jobs))

## Part A: Resume Processing

Resumes are loaded, metadata is extracted, and documents are split into section-aware chunks.

In [ ]:
rag = SimplifiedRAGSystem(chunk_size=300, chunk_overlap=50)
processed = rag.process_resumes_from_directory(str(resume_dir))
print('Processed:', processed['processed'])
print('Failed:', processed['failed'])
print('Chunks:', len(rag.db.documents))

In [ ]:
candidates = []
for resume in processed['resumes']:
    chunks = [chunk for chunk in rag.db.documents if Path(chunk['metadata']['file_path']).name == Path(resume['file']).name]
    candidates.append({'name': resume['name'], 'skills': resume['skills'], 'experience_years': resume['experience_years'], 'education': resume['education'], 'file_path': resume['file'], 'chunks': chunks, 'full_text': '\n'.join(chunk['content'] for chunk in chunks)})
print('Candidate records:', len(candidates))

## Part B: Hybrid Job Matching

Matching combines semantic and keyword relevance, then applies skill and experience filters.

In [ ]:
matcher = JobMatcher(SearchConfig(semantic_weight=0.6, keyword_weight=0.4, top_k=5, min_score_threshold=0.0, require_all_must_haves=False))
job = JobDescription(**{key: value for key, value in jobs[0].items() if key != 'id'})
matches = matcher.match_job_to_candidates(job, candidates)
print('Job:', job.title)
for rank, result in enumerate(matches[:5], 1):
    print(f'{rank}. {result.candidate_name}: {result.match_score:.2f}/100')

In [ ]:
for record in jobs:
    current_job = JobDescription(**{key: value for key, value in record.items() if key != 'id'})
    results = matcher.match_job_to_candidates(current_job, candidates)
    top = results[0] if results else None
    print(current_job.title, '->', top.candidate_name if top else 'No match')

## Explainability

The result exposes matched skills, missing skills, experience compatibility, section contributions, excerpts, and reasoning.

In [ ]:
if matches:
    result = matches[0]
    print('Candidate:', result.candidate_name)
    print('Score:', round(result.match_score, 2))
    print('Matched skills:', result.matched_skills)
    print('Missing skills:', result.missing_required_skills)
    print('Experience match:', result.experience_match)
    print('Section contributions:', result.section_contributions)
    print('Reasoning:', result.reasoning)

## Performance Metrics

Accuracy at K is the proportion of five labeled queries with at least one relevant candidate in the top five. Latency is measured with `time.perf_counter()`.

In [ ]:
metrics = run_benchmark(top_k=5)
print(json.dumps(metrics, indent=2))

## Conclusion

The complete system processes a diverse resume corpus, matches candidates against multiple jobs, explains rankings, and reports reproducible retrieval performance. The production implementation additionally supports PDF/DOCX input, Sentence Transformers, and ChromaDB.